# P1-2 · Đánh giá model đúng cách

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
X_train, y_train = make_classification(n_samples=600, n_features=20, n_informative=5, weights=[0.7], random_state=0)
model = LogisticRegression(max_iter=1000)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
print(f"F1 = {scores.mean():.3f} ± {scores.std():.3f}")   # báo cáo cả độ lệch giữa các fold, không chỉ trung bình

F1 = 0.799 ± 0.037


In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 5000))           # 5.000 đặc trưng nhiễu thuần
y = rng.integers(0, 2, size=100)           # nhãn tung đồng xu

# SAI: chọn 20 đặc trưng "tốt nhất" trên toàn bộ dữ liệu, rồi mới cross-validate
Xs = SelectKBest(f_classif, k=20).fit_transform(X, y)
cross_val_score(LogisticRegression(), Xs, y, cv=cv).mean()        # 0.87 (!)

# ĐÚNG: việc chọn đặc trưng nằm trong pipeline, chạy lại trên từng fold train
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
cross_val_score(pipe, X, y, cv=cv).mean()                          # 0.50

0.8699999999999999
0.5


## Bài tập 4: Tìm rò rỉ {khái niệm}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# (a)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y)

# (b) dự đoán giá cổ phiếu ngày mai
Xtr, Xte, ytr, yte = train_test_split(X_by_day, y_by_day, shuffle=True)

# (c) dự đoán khách có rời bỏ dịch vụ không
features = ["tuổi", "số đơn 6 tháng", "ngày hủy tài khoản"]

# (d) phân loại ảnh X-quang, mỗi bệnh nhân có 5 ảnh
Xtr, Xte = train_test_split(all_images, test_size=0.2)

## Bài tập 5: Chọn ngưỡng tối ưu F1 {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# dữ liệu mất cân bằng để thử: 10% lớp dương
X, y = make_classification(n_samples=3000, n_features=20, n_informative=6, weights=[0.9], flip_y=0.03, random_state=1)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=0)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)

def best_threshold(y_val, proba_val):
    ths = np.arange(0.05, 0.96, 0.01)
    scores = [f1_score(y_val, proba_val >= t) for t in ths]
    i = int(np.argmax(scores))
    return ths[i], scores[i]

t, f = best_threshold(y_val, model.predict_proba(X_val)[:, 1])
p_test = model.predict_proba(X_test)[:, 1]
print(f"ngưỡng {t:.2f}: F1 test {f1_score(y_test, p_test >= t):.3f} | ngưỡng 0.5: {f1_score(y_test, p_test >= 0.5):.3f}")

ngưỡng 0.25: F1 test 0.681 | ngưỡng 0.5: 0.579
